In [8]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from lightning import pytorch as pl
from rdkit import Chem
from chemprop import data, featurizers, models, nn, utils




In [10]:
# ==========================================
# 1. SETUP, CONFIGURATION & DATA CLEANING
# ==========================================
data_dir = Path.cwd()
smiles_col = "smiles"   
target_col = "activity" 
num_workers = 4         

print("Loading and cleaning datasets...")

# --- PRE-TRAINING DATA ---
# Dataset 1: Has 'SMILE' and 'MIC'
df_1 = pd.read_excel(data_dir / "Dataset_1.xlsx")
df_1 = df_1.rename(columns={"SMILE": "smiles"})
# CLEANING FIX: Strip out '>' and '<' symbols and convert to numeric
df_1["MIC"] = pd.to_numeric(df_1["MIC"].astype(str).str.replace('>', '').str.replace('<', '').str.strip(), errors='coerce')
df_1["activity"] = (df_1["MIC"] <= 16).astype(float)
df_1 = df_1[["smiles", "activity"]]

# Dataset 2: Has no headers. 
df_2 = pd.read_excel(data_dir / "Dataset_2.xlsx", header=None)
df_2 = df_2.rename(columns={1: "smiles", 2: "MIC"})
# CLEANING FIX: Strip out '>' and '<' symbols and convert to numeric
df_2["MIC"] = pd.to_numeric(df_2["MIC"].astype(str).str.replace('>', '').str.replace('<', '').str.strip(), errors='coerce')
df_2["activity"] = (df_2["MIC"] <= 16).astype(float)
df_2 = df_2[["smiles", "activity"]]

# Dataset 3: Has PUBCHEM columns
df_3 = pd.read_excel(data_dir / "Dataset_3.xlsx")
df_3 = df_3.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "smiles"})
df_3["activity"] = df_3["PUBCHEM_ACTIVITY_OUTCOME"].map({"Active": 1.0, "Inactive": 0.0})
df_3 = df_3[["smiles", "activity"]]

# Combine all pre-training data cleanly and drop any rows with missing values
df_pre = pd.concat([df_1, df_2, df_3], ignore_index=True).dropna()


# --- FINE-TUNING DATA ---
# Dataset 4: Has PUBCHEM columns
df_fine = pd.read_excel(data_dir / "Dataset_4.xlsx")
df_fine = df_fine.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "smiles"})
df_fine["activity"] = df_fine["PUBCHEM_ACTIVITY_OUTCOME"].map({"Active": 1.0, "Inactive": 0.0})
df_fine = df_fine[["smiles", "activity"]].dropna()


# --- TEST SET DATA ---
# Dataset 5: Has PUBCHEM columns
df_test = pd.read_csv(data_dir / "Dataset_5.csv")
df_test = df_test.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "smiles"})
df_test["activity"] = df_test["PUBCHEM_ACTIVITY_OUTCOME"].map({"Active": 1.0, "Inactive": 0.0})
df_test = df_test[["smiles", "activity"]].dropna()

print(f"Pre-training set size: {len(df_pre)} molecules")
print(f"Fine-tuning set size: {len(df_fine)} molecules")
print(f"Test set size: {len(df_test)} molecules")

Loading and cleaning datasets...
Pre-training set size: 9937 molecules
Fine-tuning set size: 2809 molecules
Test set size: 892 molecules


In [2]:
df_pre1

,No,SMILE,MIC
0,1,COC1=C(OC)C=CC(C1=C2)=CC3=[N+]2CCC4=CC(OC/C=C(...,4
1,2,COC1=C(OC)C=CC(C1=C2)=CC3=[N+]2CCC4=CC(OCC5CCC...,2
2,3,COC1=C(OC)C=CC(C1=C2)=CC3=[N+]2CCC4=CC(OC/C=C/...,8
3,4,COC1=C(OC)C2=C[N+]3=C(C=C2C=C1)C4=CC(OC)=C(OCC...,8
4,5,COC1=C(OC)C=CC(C1=C2)=CC3=[N+]2CCC4=CC(OCCCCC#...,4
...,...,...,...
132,133,COC(C1=C[N+](CCC2=CC(OCO3)=C3C=C42)=C4C=C1C=C5...,＞16
133,134,COC1=C(OCCOCCNCC#C)C2=C[N+](CCC3=CC(OCO4)=C4C=...,＞16
134,135,COC1=C(OCCCCCC#C)C(C[N+]2(CCCCCC#C)CCC3=CC(OCO...,2
135,136,COC1=C(OCCOCCN)C2=C[N+](CCC3=CC(OCO4)=C4C=C53)...,＞16


In [3]:
df_pre2

,CHEMBL1082,CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)c3ccc(O)cc3)C(=O)N2[C@H]1C(=O)O,'=',0.8,ug/mL
0,CHEMBL116438,COc1cc(/C=C/C(=O)/C=C(O)/C=C/c2ccc(O)c(OC)c2)c...,'=',15.000,ug/mL
1,CHEMBL1209348,CCOC(=O)c1ccc(NC(=O)c2cc3ccc(O)cc3oc2=O)cc1,'=',8.000,ug/mL
2,CHEMBL1209408,CCOC(=O)c1ccc(NC(=O)c2cc3cc(C)ccc3oc2=O)cc1,'=',4.000,ug/mL
3,CHEMBL1209409,CCOC(=O)c1ccc(NC(=O)c2cc3cc(Br)cc(Cl)c3oc2=O)cc1,'=',8.000,ug/mL
4,CHEMBL1209410,CCOC(=O)c1ccc(NC(=O)c2cc3cc(Br)cc(Br)c3oc2=O)cc1,'=',64.000,ug/mL
...,...,...,...,...,...
795,CHEMBL99389,COc1ccccc1C/N=C(\N)Nc1nc(-c2ccc(CNC(C)=O)o2)cs1,'=',0.035,ug/mL
796,CHEMBL99460,COc1ccccc1OCC/N=C(\N)Nc1nc(-c2ccc(CNC(C)=O)o2)cs1,'=',2.500,ug/mL
797,CHEMBL99510,COc1ccc(C/N=C(\N)Nc2nc(-c3ccc(CNC(C)=O)o3)cs2)cc1,'=',0.280,ug/mL
798,CHEMBL99720,CC(=O)NCc1ccc(-c2csc(NC(=N)NCc3ccccc3)n2)o1,'=',0.085,ug/mL


In [4]:
df_pre3

,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME
0,135651320,CC1=CC=C(C=C1)S(=O)(=O)N/N=C/C2=CNC3=C2C=C(C=C...,Active
1,650194,CC1=CC(=C(C=C1)NC2=NCC(=O)N2CC3=CC4=C(C=C3)OCO4)C,Inactive
2,650260,C1=C(C=C(C2=C1NC(=O)C(=O)N2)Cl)Cl,Inactive
3,645565,CC1=CC(=C(C=C1)C)N2C(=NN=C2SCC(=O)N)CNC3=CC=C(...,Inactive
4,648171,CC(=C)CN1C2=C(N=C1NCCOC)N(C(=O)NC2=O)C,Inactive
...,...,...,...
8994,1799133,CC1=C(C(=S)N(N1)C2=CC=CC=C2)C=NCCCN=CC3=C(NN(C...,Active
8995,3323365,CC1=NN(C(=C1NC(=O)OCC2=C(C=C(C=C2)Cl)Cl)Cl)C,Active
8996,644973,C1CN(C2=CC=CC=C21)C3=NC=NC4=C3N=NN4CC5=CC=CC=C5,Inactive
8997,650125,CC1=CC(=NC(=N1)N(CC(=O)NC2=CC=CC=C2)C#N)C,Inactive


In [6]:
df_fine

,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME
0,9551376,CC1=CC=C(C=C1)CSC2=NC3=C(N2CC4=C(OC(=N4)C5=CC(...,Inactive
1,38258,C(NC(=O)NC1C(=O)NC(=O)N1CO)NC(=O)NC2C(=O)NC(=O...,Inactive
2,2957586,C1=CC(=CC=C1NC(=O)NC2=CC(=C(C=C2)OC(F)F)Cl)[N+...,Active
3,670460,C1OC2=C(O1)C=C(C=C2)C(=O)NN=CC3=C(C=CC4=CC=CC=...,Active
4,56704,CC(=O)OCC(=O)NCCCOC1=CC=CC(=C1)CN2CCCCC2.Cl,Inactive
...,...,...,...
2804,3911449,COC1=CC=C(C=C1)C(=O)NNC(=S)NC(=O)C2=CC(=CC=C2)...,Inactive
2805,2533856,CC1=C(C(=NN1C2=CC=CC=C2)C)C(=O)NC3=NN=C(O3)C4=...,Inactive
2806,2457974,CCOC1=CC2=C(C=C1)N=C(S2)NC(=O)C3=NC=C(N=C3)C,Inactive
2807,6416438,CC1=NN=C(C2=CC=CC=C12)NCCC3=CC=CC=C3,Active


In [7]:
df_test

,Unnamed: 0,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME
0,1,901425,C1=C(OC(=C1)[N+](=O)[O-])C2=CC=NN2,Active
1,2,4640068,CCN1C=C(C(=O)C2=CC(=C(C=C21)N3CCN(CC3)C(=S)NC(...,Active
2,3,149096,C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)...,Active
3,4,54684459,C[C@@]1(C2C[C@H]3[C@@H](C(=O)C(=C([C@]3(C(=O)C...,Active
4,5,6915944,C=CC1=C(N2[C@@H]([C@@H](C2=O)NC(=O)/C(=N\O)/C3...,Active
...,...,...,...,...
887,888,9550753,CN1C=C(C2=CC=CC=C21)CC(=O)N3CCN(CC3)C4=CC=CC=C4F,Inactive
888,889,17429903,CC1=C(C=C(C=C1)NC(=O)C2=CC(=CN2C)S(=O)(=O)N3CC...,Inactive
889,890,49792495,COC1=CC=CC(=C1)C2=CN=C(N2)C(C3=CC(=CC=C3)OC)O,Inactive
890,891,3648794,C1=CC=C(C=C1)CNC(=O)C2=C3NC(=O)C4=CC=CC=C4N3C(...,Inactive
